# Cặp tương tác, tần suất và biến dẫn xuất

Bước 10 của kế hoạch S6E11.

P1, P2 và P3 lần lượt mở rộng GLM với 4, 8 và 13 khóa tương tác. Ngưỡng chia phân vị được học trong mỗi tập tham chiếu. COUNT thêm log(1+tần suất) từ dữ liệu huấn luyện. DERIVED đo riêng bình phương số cờ và tỷ lệ số tiền trên số dư.

## Chạy lại

Chạy từ thư mục `plan_runs` với môi trường Python đã dùng cho RTX 4060. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```powershell
& '..\.build\venv\Scripts\python.exe' -B -X utf8 run_plan.py --stage features
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'features']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `DERIVED` | `T04` | 0.948542 | -0.000032 | 0.520315 | 0.044972 | 809.5 | 0.947170 | 0.950143 | 0.953734 | 0.949127 | 0.942875 |
| `P1-P2` | `P1` | 0.947581 | +0.000097 | 0.515491 | 0.045286 | 40.6 | 0.946502 | 0.948846 | 0.953047 | 0.947864 | 0.941744 |
| `P1` | `T01` | 0.947484 | +0.000223 | 0.515070 | 0.045314 | 26.3 | 0.946547 | 0.949016 | 0.952914 | 0.947636 | 0.941467 |
| `P1-P2-P3` | `P1-P2` | 0.947385 | -0.000196 | 0.514208 | 0.045363 | 42.6 | 0.946229 | 0.948721 | 0.952837 | 0.947988 | 0.941226 |
| `COUNT` | `T04` | 0.946825 | -0.001749 | 0.505820 | 0.046290 | 702.9 | 0.945270 | 0.949068 | 0.951247 | 0.946996 | 0.941942 |

## Đọc dữ liệu kết quả hiện tại

Cell này xác minh những lượt hoàn tất từ dự đoán OOF và tệp metric.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import final_summary, read_stage

data_dir = plan_dir / "results" / "full"
rows, pending, notes = read_stage('03-pairs-count', data_dir)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(data_dir, markdown_links=False)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    directory = data_dir / record["name"]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    with np.load(directory / "predictions.npz", allow_pickle=False) as saved:
        display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

Chênh lệch của P2 và P3 đo cả nhóm khóa vừa thêm. Cần phép bỏ từng khóa nếu muốn quy kết lợi ích cho một tương tác riêng.